# Agent-as-a-Judge with a Custom Provider

This notebook extends [nps_agent.ipynb](./nps_agent.ipynb) to demonstrate two additional capabilities:

1. **Custom judge provider** — Use a local model (`ollama/gpt-oss:20b`) served through LlamaStack as the judge, instead of the default OpenAI provider.
2. **Judge tracing** — Wrap the judge evaluation in an `mlflow.start_span` context so the judge's own LLM calls appear as child spans under a single parent trace.

**Prerequisites:**
- Ollama running on `localhost:11434` with the model pulled:  
  `ollama pull gpt-oss:20b`
- LlamaStack server on `localhost:8321`:  
  `OLLAMA_URL=http://localhost:11434/v1 llama stack run starter`
- `OPENAI_API_KEY` in environment (used by the main NPS agent)
- `NPS_API_KEY` in environment (for the NPS MCP server)
- Python packages: `openai-agents`, `mlflow`, `llama-stack-client`, `python-dotenv`

> **Reference:** 
- [MLflow Agent-as-a-Judge docs](https://mlflow.org/docs/latest/genai/eval-monitor/scorers/llm-judge/agentic-overview/) 
- [MLflow Manual Tracing docs](https://mlflow.org/docs/latest/genai/tracing/app-instrumentation/manual-tracing/)

In [1]:
import os
from dotenv import load_dotenv

# Load .env from parent directory (agents_tracing-eval_mlflow/.env)
env_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
load_dotenv(env_path)

import mlflow
from mlflow.entities import SpanType
from mlflow.genai.judges import make_judge
from typing import Literal

# Auto-instrument OpenAI Agents SDK calls
mlflow.openai.autolog()

/Users/rranabha/red_hat_repos/agents/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

We configure two models:
- **Main agent model** — `gpt-4o` via OpenAI (answers user questions)
- **Judge model** — `ollama/gpt-oss:20b` via LlamaStack (evaluates the agent's trace)

The judge model uses `openai` as the provider name because LlamaStack exposes an OpenAI-compatible API.  
MLflow routes requests through [litellm](https://docs.litellm.ai/), which uses the OpenAI client SDK to talk to LlamaStack.

In [2]:
# Main agent configuration
MAIN_AGENT_MODEL = "gpt-4o"
LLAMA_STACK_URL = "http://localhost:8321/v1/"

# Judge model configuration
# Since LlamaStack is OpenAI-compatible, we use "openai" as the provider.
JUDGE_PROVIDER = "openai"
JUDGE_MODEL_ID = "ollama/gpt-oss:20b"

# LlamaStack lets you easily swap judge models:
#   JUDGE_MODEL_ID = "openai/gpt-4o"              # OpenAI via LlamaStack
#   JUDGE_MODEL_ID = "vllm/openai/gpt-oss-120b"   # remote vLLM

JUDGE_MODEL = f"{JUDGE_PROVIDER}:/{JUDGE_MODEL_ID}"
print(f"Judge model: {JUDGE_MODEL}")

Judge model: openai:/ollama/gpt-oss:20b


In [3]:
db_path = os.path.join(os.getcwd(), "mlflow.db")
mlflow.set_tracking_uri(f"sqlite:///{db_path}")
mlflow.set_experiment("nps-agent")
print(f"MLflow database: {db_path}")

2026/03/04 10:56:55 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/03/04 10:56:55 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/03/04 10:56:55 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/03/04 10:56:55 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/03/04 10:56:55 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/03/04 10:56:55 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/03/04 10:56:55 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/03/04 10:56:55 INFO alembic.runtime.migration: Will assume non-transactional DDL.


MLflow database: /Users/rranabha/red_hat_repos/agents/examples/agents_tracing-eval_mlflow/nps_agent/mlflow.db


## Agent Function

Runs the NPS agent using the [OpenAI Agents SDK](https://github.com/openai/openai-agents-python) with MCP tools attached.  
`mlflow.openai.autolog()` (set above) automatically traces the agent's execution.

In [4]:
from agents import Agent, Runner, set_default_openai_client
from agents.mcp import MCPServerStdio
from openai import AsyncClient

AGENT_INSTRUCTIONS = (
    "You are a helpful National Parks Service assistant. "
    "Use the available tools to answer questions about national parks, "
    "events, activities, campgrounds, and visitor information. "
)


async def run_nps_agent(prompt: str) -> str:
    """Run the NPS agent with MCP tools and return the text response."""
    command = "uv"
    args = ["run", "fastmcp", "run", "./nps_mcp_server.py"]
    env = {**os.environ, "NPS_API_KEY": os.environ.get("NPS_API_KEY", "")}

    async with MCPServerStdio(params={"command": command, "args": args, "env": env}) as mcp_server:
        async_client = AsyncClient(
            base_url=os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1"),
            api_key=os.environ.get("OPENAI_API_KEY", ""),
        )
        set_default_openai_client(client=async_client)

        agent = Agent(
            name="NPS Agent",
            instructions=AGENT_INSTRUCTIONS,
            mcp_servers=[mcp_server],
            model=MAIN_AGENT_MODEL,
        )

        result = await Runner.run(agent, prompt)
        return result.final_output

## Agent-as-a-Judge

An Agent-as-a-Judge evaluates the agent's trace *after* execution. Instead of just looking at inputs/outputs, it uses tools to inspect the full execution:
- What spans were created
- What tools were called
- How long each step took

The `{{ trace }}` template variable in the instructions tells MLflow to provide the judge with trace-inspection tools (e.g., `GetTraceInfo`, `ListSpans`, `GetSpan`, `SearchTraceRegex`).

The `inference_params` tell MLflow/litellm to route judge requests to our LlamaStack endpoint instead of the default OpenAI API.

In [5]:
nps_judge = make_judge(
    name="nps_agent_evaluator",
    instructions=(
        "Evaluate the NPS agent's performance in {{ trace }}.\n\n"
        "Check for:\n"
        "1. Response Quality: Did the agent correctly identify parks and provide accurate information?\n"
        "2. Tool Usage: Were the correct NPS MCP tools used (search_parks, get_park_events, etc.)?\n"
        "3. Completeness: Did the agent answer all parts of the user's question?\n\n"
        "Rate as: 'good', 'acceptable', or 'poor'"
    ),
    feedback_value_type=Literal["good", "acceptable", "poor"],
    model=JUDGE_MODEL,
    inference_params={
        "api_base": LLAMA_STACK_URL,
        "api_key": "fake-key",  # LlamaStack isn't setup to require a real key
    },
)

## Run Agent & Evaluate

1. Send a query to the NPS agent
2. Retrieve the MLflow trace from the execution
3. Pass the trace to the judge for evaluation
4. View the judge's verdict and rationale

In [6]:
prompt = "What campgrounds are available at the Grand Canyon?"
result = await run_nps_agent(prompt)

print(f"\nAgent response:\n{result}")
print("-" * 60)


Agent response:
Here are the campgrounds available at the Grand Canyon:

1. **Desert View Campground (Reservations Required)**
   - **Description:** Offers 49 campsites near the East Entrance in a peaceful setting. Maximum RV length is 30 feet.
   - **Reservations:** Required, available through [Recreation.gov](https://www.recreation.gov/camping/campgrounds/258825).
   - **Season:** April 11 through October 11, 2026.

2. **Mather Campground - South Rim**
   - **Description:** Located in Grand Canyon Village, this campground has 327 sites with campfire rings, picnic tables, and flush toilets. No hookups available.
   - **Reservations:** Required for most sites, available through [Recreation.gov](https://www.recreation.gov/camping/campgrounds/232490). 15 first-come, first-served sites are also available.
   - **Availability:** March 1 through November 30.

3. **Trailer Village RV Park - South Rim**
   - **Description:** The only RV campground with full hookups, including sewage, water, 

In [7]:
# Retrieve the trace from the agent execution
trace_id = mlflow.get_last_active_trace_id()
trace = mlflow.get_trace(trace_id)
print(f"Trace ID: {trace_id}")

Trace ID: tr-6c4e0077e4e028d0c29f635c22f493ec


### Tracing the Judge

By wrapping the judge call in `mlflow.start_span(...)`, the judge's internal LLM calls
appear as child spans under a single parent span in the MLflow UI. Without this,
each internal LLM request would appear as a separate top-level trace.
(Comparison shown in figure below)

In [8]:
# Wrap judge evaluation in a span so all its LLM calls appear as children
with mlflow.start_span(name="agent-judge-evaluation", span_type=SpanType.CHAIN) as span:
    span.set_inputs({"trace_id": trace_id})
    feedback = nps_judge(trace=trace)
    span.set_outputs({"verdict": feedback.value, "rationale": feedback.rationale})

print(f"Verdict:   {feedback.value}")
print(f"Rationale: {feedback.rationale}")

Verdict:   good
Rationale: The agent correctly identified the three campgrounds at Grand Canyon National Park—Desert View Campground, Mather Campground, and Trailer Village RV Park—providing accurate descriptions, reservation details, and URLs. It used the appropriate NPS MCP tool `get_park_campgrounds` to retrieve the information, and the answer fully addresses the user’s question with no omitted parts.


### Judge Trace in MLflow UI

Below is an example of what the judge trace looks like in the MLflow UI. Notice how all of the judge agent's individual LLM calls are grouped as child spans under the single `agent-judge-evaluation` parent span:

![Judge Agent Traces](./images/judge-agent-traces.png)

## Next Steps

- **Local LLMs as judges** — Evaluate the performance/cost tradeoff of smaller local models as judges vs. cloud models.
- **Richer span metadata** — The judge selects which spans to inspect based on span metadata. Providing descriptive span names and attributes helps the judge make better evaluations.
- **Multi-turn conversations** — Test how the judge handles complex conversation histories (e.g., when the user instructs the agent to skip tools):
  ```python
  prompt = [
      {"role": "user", "content": "Please just answer without using tools."},
      {"role": "assistant", "content": "Sure, I'll answer directly."},
      {"role": "user", "content": "What campgrounds are available at the Grand Canyon?"},
  ]
  ```

## View Traces in MLflow UI

Start the MLflow UI to view traces and assessments:

```bash
mlflow ui --port 5001
```

Then open http://localhost:5001 in your browser.

### How to Navigate

1. **Select the Experiment** — Click on `nps-agent` in the left sidebar
2. **Go to Traces tab** — Click the "Traces" tab to see all agent executions
3. **View Trace Details** — Click on any Trace ID to open the trace detail view
   - You'll see the span hierarchy showing the agent's execution
   - Click on individual spans to see inputs/outputs for each step
4. **View Assessments** — In the trace detail view, look for the assessments side-panel on the right
   - This shows the Agent-as-a-Judge evaluation results